# Getting Started with ragbandit-core

Full pipeline walkthrough — each cell corresponds to one stage of the
document-processing pipeline.  Run the **Setup** cell first, then execute
cells top-to-bottom.

In [ ]:
# ── Change these to use your own document and API keys ──
PDF_PATH = "../tests/fixtures/sample.pdf"
ENV_PATH = "../tests/.env"
# ────────────────────────────────────────────────────────

import os, logging
from dotenv import load_dotenv

load_dotenv(ENV_PATH)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)

MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY")
print("Setup complete ✓")

## 1 — Build the pipeline

We assemble a `DocumentPipeline` with MistralOCR, two refiners,
a FixedSizeChunker, and MistralEmbedder.

In [ ]:
from ragbandit.documents import (
    DocumentPipeline,
    MistralOCR,
    ReferencesRefiner,
    FootnoteRefiner,
    FixedSizeChunker,
    MistralEmbedder,
)

pipeline = DocumentPipeline(
    ocr_processor=MistralOCR(api_key=MISTRAL_API_KEY),
    refiners=[
        ReferencesRefiner(api_key=MISTRAL_API_KEY),
        FootnoteRefiner(api_key=MISTRAL_API_KEY),
    ],
    chunker=FixedSizeChunker(chunk_size=1000, overlap=200),
    embedder=MistralEmbedder(api_key=MISTRAL_API_KEY, model="mistral-embed"),
)
print("Pipeline ready")

## 2 — OCR

Extract text from each page of the PDF.

In [ ]:
ocr_result = pipeline.run_ocr(PDF_PATH)

print(f"Pages extracted: {len(ocr_result.pages)}")
print(f"OCR model     : {ocr_result.model}")
for i, page in enumerate(ocr_result.pages[:3]):
    preview = page.markdown[:150].replace('\n', ' ')
    print(f"\nPage {i} preview: {preview}...")

## 3 — Refiners

Run the ReferencesRefiner and FootnoteRefiner to clean up the OCR output.

In [ ]:
refining_results = pipeline.run_refiners(ocr_result)

for rr in refining_results:
    print(f"Refiner: {rr.component_name}")
    print(f"  Pages: {len(rr.pages)}")
    if rr.extracted_data:
        for key, val in rr.extracted_data.items():
            preview = str(val)[:120].replace('\n', ' ')
            print(f"  {key}: {preview}")
    print()

final_doc = refining_results[-1]

## 4 — Chunking

Split the refined document into fixed-size text chunks.

In [ ]:
chunk_result = pipeline.run_chunker(final_doc)

chunks = chunk_result.chunks
sizes = [len(c.text) for c in chunks]

print(f"Total chunks : {len(chunks)}")
print(f"Min size     : {min(sizes)} chars")
print(f"Max size     : {max(sizes)} chars")
print(f"Avg size     : {sum(sizes) // len(sizes)} chars")

for i, c in enumerate(chunks[:3]):
    preview = c.text[:120].replace('\n', ' ')
    print(f"\nChunk {i} [{len(c.text)} chars]: {preview}...")

## 5 — Embedding

Generate vector embeddings for each chunk.

In [ ]:
embedding_result = pipeline.run_embedder(chunk_result)

embs = embedding_result.chunks_with_embeddings
dim = len(embs[0].embedding) if embs else 0

print(f"Vectors    : {len(embs)}")
print(f"Dimensions : {dim}")
print(f"Model      : {embedding_result.model_name}")

if embs:
    print(f"\nFirst vector (first 5 dims): {embs[0].embedding[:5]}")

## Summary

Print a final overview of what the pipeline produced.

In [ ]:
print("Pipeline complete!")
print(f"  OCR pages  : {len(ocr_result.pages)}")
print(f"  Refiners   : {len(refining_results)}")
print(f"  Chunks     : {len(chunks)}")
print(f"  Embeddings : {len(embs)} x {dim}d")